# Phase 4 — Layer-wise Probing: Where Each Emotion Crystallizes

Train linear probes (logistic regression) on the [CLS] token at each of BERT's
13 hidden states (embedding + 12 encoder layers) for every active emotion.

**Goal.** Localise the depth at which each emotion becomes linearly separable.

**Pipeline.**
- **Setup** — paths, imports, helpers.
- **Part A** — extract hidden states + train probes.
- **Part B** — crystallization map, trajectories, correlations, information gain.
- **Part C** — (optional) link to compression sensitivity from the lesion study.
- **Part D** — export every result as CSV to `results/notebook4/`.

All computations use the 23 active emotions (5 excluded from GoEmotions).

## Setup

In [ ]:
import sys, os, time, warnings
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/transformer-structural-compression-v2'
    IN_COLAB = True
except ImportError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    IN_COLAB = False

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from transformers import AutoModelForSequenceClassification

from src.data import load_goemotions

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 200, 'font.size': 11})

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
EXPORT_DIR = os.path.join(RESULTS_DIR, 'notebook4')
os.makedirs(EXPORT_DIR, exist_ok=True)

MODEL_SUBDIR = 'bert-goemotions-23emo-final'
MODEL_PATH = os.path.join(RESULTS_DIR, MODEL_SUBDIR)

EXCLUDED_EMOTIONS = ['neutral', 'grief', 'nervousness', 'pride', 'relief']
N_HIDDEN_LAYERS = 13  # embedding + 12 encoder layers
CRYSTAL_THRESHOLD = 0.80  # fraction of max F1 that defines crystallization
BATCH_SIZE = 64

print(f'Project root: {PROJECT_ROOT}')
print(f'Device:       {DEVICE}')
print(f'Export dir:   {EXPORT_DIR}')

In [ ]:
@torch.no_grad()
def extract_all_hidden_states(model, dataset, data_collator, batch_size=BATCH_SIZE):
    """Extract [CLS] hidden states from all 13 layers for every example."""
    model.eval()
    dev = next(model.parameters()).device

    all_hidden = [[] for _ in range(N_HIDDEN_LAYERS)]
    all_labels = []
    n_batches = (len(dataset) + batch_size - 1) // batch_size
    t0 = time.time()

    for i in range(0, len(dataset), batch_size):
        batch = data_collator([dataset[j] for j in range(i, min(i + batch_size, len(dataset)))])
        outputs = model(
            input_ids=batch['input_ids'].to(dev),
            attention_mask=batch['attention_mask'].to(dev),
            output_hidden_states=True,
        )
        for layer_idx in range(N_HIDDEN_LAYERS):
            cls = outputs.hidden_states[layer_idx][:, 0, :].cpu().numpy()
            all_hidden[layer_idx].append(cls)
        all_labels.append(batch['labels'].numpy())

        if (i // batch_size + 1) % 50 == 0:
            done = i // batch_size + 1
            print(f'  Batch {done}/{n_batches} ({time.time()-t0:.0f}s)')

    hidden_states = [np.concatenate(h, axis=0) for h in all_hidden]
    labels = np.concatenate(all_labels, axis=0)
    print(f'  Done: {hidden_states[0].shape[0]} examples in {time.time()-t0:.0f}s')
    return hidden_states, labels


def train_all_probes(hidden_train, labels_train, hidden_test, labels_test):
    """Train linear probes for every (layer, emotion) pair."""
    n_layers = len(hidden_train)
    n_emotions = labels_train.shape[1]
    probe_f1 = np.zeros((n_emotions, n_layers))

    t0 = time.time()
    for layer_idx in range(n_layers):
        scaler = StandardScaler()
        X_train = scaler.fit_transform(hidden_train[layer_idx])
        X_test = scaler.transform(hidden_test[layer_idx])

        for emo_idx in range(n_emotions):
            y_train = labels_train[:, emo_idx].astype(int)
            y_test = labels_test[:, emo_idx].astype(int)
            if y_train.sum() < 5 or y_test.sum() < 5:
                probe_f1[emo_idx, layer_idx] = 0.0
                continue
            clf = LogisticRegression(max_iter=500, random_state=42, C=1.0, solver='lbfgs')
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)
            probe_f1[emo_idx, layer_idx] = f1_score(y_test, y_pred, zero_division=0)

        layer_name = 'Emb' if layer_idx == 0 else f'L{layer_idx - 1}'
        print(f'  {layer_name:>4s}: mean F1 = {probe_f1[:, layer_idx].mean():.4f}  ({time.time()-t0:.0f}s)')

    return probe_f1


def compute_crystallization_layer(probe_f1, threshold=CRYSTAL_THRESHOLD):
    """Return first layer at which each emotion reaches threshold x max F1."""
    n_emotions, n_layers = probe_f1.shape
    crystal = np.full(n_emotions, n_layers - 1, dtype=int)
    for i in range(n_emotions):
        max_f1 = probe_f1[i].max()
        if max_f1 < 0.01:
            continue
        target = threshold * max_f1
        above = np.where(probe_f1[i] >= target)[0]
        if len(above) > 0:
            crystal[i] = above[0]
    return crystal


LAYER_LABELS = ['Emb'] + [f'L{i}' for i in range(12)]

## Part A — Hidden State Extraction and Probe Training

Extract [CLS] hidden states from both train and test splits at every layer,
then train 13 x 23 = 299 logistic-regression probes.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH, problem_type='multi_label_classification',
)
model.eval().to(DEVICE)
print(f'Model loaded from {MODEL_PATH}')

train_ds, val_ds, test_ds, emotion_names, data_collator = load_goemotions(
    exclude_emotions=EXCLUDED_EMOTIONS
)
num_labels = len(emotion_names)
print(f'Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}')
print(f'Emotions ({num_labels}): {emotion_names}')

In [ ]:
print('Extracting hidden states from TRAIN set...')
hidden_train, labels_train = extract_all_hidden_states(model, train_ds, data_collator)
print(f'  Shape per layer: {hidden_train[0].shape}')

print('\nExtracting hidden states from TEST set...')
hidden_test, labels_test = extract_all_hidden_states(model, test_ds, data_collator)
print(f'  Shape per layer: {hidden_test[0].shape}')

model.cpu()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
print('\nModel moved to CPU, GPU memory freed.')

In [ ]:
print(f'Training {N_HIDDEN_LAYERS * num_labels} probing classifiers...')
print('=' * 50)
probe_f1 = train_all_probes(hidden_train, labels_train, hidden_test, labels_test)
print(f'\nProbe matrix shape: {probe_f1.shape}  (emotions x layers)')

## Part B — Crystallization Analysis

### B.1 — Crystallization Map

Probe F1 heatmap. Rows sorted by crystallization layer (first layer that
reaches 80% of the maximum F1 for that emotion).

In [ ]:
crystal_layer = compute_crystallization_layer(probe_f1)
max_f1_per_emo = probe_f1.max(axis=1)

sort_idx = np.lexsort((max_f1_per_emo, crystal_layer))
sorted_emotions = [emotion_names[i] for i in sort_idx]
sorted_f1 = probe_f1[sort_idx]

fig, ax = plt.subplots(figsize=(12, 14))
sns.heatmap(
    sorted_f1, annot=True, fmt='.2f', cmap='RdYlGn',
    xticklabels=LAYER_LABELS, yticklabels=sorted_emotions,
    vmin=0, vmax=sorted_f1.max() + 0.05,
    linewidths=0.5, linecolor='white', ax=ax,
    cbar_kws={'label': 'Probe F1', 'shrink': 0.6},
)
for i, orig_idx in enumerate(sort_idx):
    cl = crystal_layer[orig_idx]
    ax.add_patch(plt.Rectangle((cl, i), 1, 1, fill=False, edgecolor='black', linewidth=2.5))

ax.set_xlabel('Layer', fontsize=13)
ax.set_title('Where Each Emotion Crystallizes in BERT\n'
             '(black box = crystallization layer: first layer at 80% of max F1)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'probing_crystallization_map.png'), bbox_inches='tight')
plt.show()

### B.2 — Formation Trajectories

In [ ]:
valid_mask = max_f1_per_emo > 0.01
valid_indices = np.where(valid_mask)[0]

earliest = valid_indices[np.argsort(crystal_layer[valid_indices])[:2]]
latest = valid_indices[np.argsort(crystal_layer[valid_indices])[-2:]]
highest = valid_indices[np.argsort(max_f1_per_emo[valid_indices])[-2:]]
lowest = valid_indices[np.argsort(max_f1_per_emo[valid_indices])[:2]]

selected = sorted(set(np.concatenate([earliest, latest, highest, lowest])),
                  key=lambda i: crystal_layer[i])

fig, ax = plt.subplots(figsize=(14, 7))
cmap = plt.cm.tab10
for i, emo_idx in enumerate(selected):
    ax.plot(range(N_HIDDEN_LAYERS), probe_f1[emo_idx], 'o-', color=cmap(i % 10),
            label=f'{emotion_names[emo_idx]} (crystal: {LAYER_LABELS[crystal_layer[emo_idx]]})',
            linewidth=2.5, markersize=6)

ax.axvspan(0, 4.5, alpha=0.05, color='blue')
ax.axvspan(4.5, 8.5, alpha=0.05, color='green')
ax.axvspan(8.5, 12.5, alpha=0.05, color='red')
ax.text(2, ax.get_ylim()[1] * 0.95, 'Early', ha='center', fontsize=10, color='blue', alpha=0.5)
ax.text(6.5, ax.get_ylim()[1] * 0.95, 'Middle', ha='center', fontsize=10, color='green', alpha=0.5)
ax.text(10.5, ax.get_ylim()[1] * 0.95, 'Late', ha='center', fontsize=10, color='red', alpha=0.5)

ax.set_xticks(range(N_HIDDEN_LAYERS))
ax.set_xticklabels(LAYER_LABELS)
ax.set_xlabel('Layer', fontsize=13)
ax.set_ylabel('Probe F1', fontsize=13)
ax.set_title('Emotion Formation Trajectories Through BERT', fontsize=15, fontweight='bold')
ax.legend(fontsize=10, loc='center left', bbox_to_anchor=(1.02, 0.5))
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'probing_trajectories.png'), bbox_inches='tight')
plt.show()

### B.3 — Crystallization Depth Bar Chart

In [ ]:
valid_emos = [(emotion_names[i], int(crystal_layer[i]), float(max_f1_per_emo[i]))
              for i in range(num_labels) if valid_mask[i]]
valid_emos.sort(key=lambda x: (x[1], -x[2]))

colors = []
for _, cl, _ in valid_emos:
    if cl <= 4:
        colors.append('#2ecc71')
    elif cl <= 8:
        colors.append('#f39c12')
    else:
        colors.append('#e74c3c')

fig, ax = plt.subplots(figsize=(12, 8))
y_pos = range(len(valid_emos))
ax.barh(y_pos, [cl for _, cl, _ in valid_emos], color=colors, edgecolor='white')
ax.set_yticks(y_pos)
ax.set_yticklabels([e for e, _, _ in valid_emos], fontsize=10)
ax.set_xlabel('Crystallization Layer', fontsize=13)
ax.set_title('When Each Emotion Becomes Linearly Separable', fontsize=14, fontweight='bold')
ax.set_xticks(range(N_HIDDEN_LAYERS))
ax.set_xticklabels(LAYER_LABELS)
ax.invert_yaxis()

legend_elements = [
    Patch(facecolor='#2ecc71', label='Early (Emb-L3)'),
    Patch(facecolor='#f39c12', label='Middle (L4-L7)'),
    Patch(facecolor='#e74c3c', label='Late (L8-L11)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'probing_crystallization_bars.png'), bbox_inches='tight')
plt.show()

### B.4 — What Drives Crystallization Depth?

Two correlations:
- Training frequency vs crystallization layer.
- Crystallization layer vs maximum probe F1.

In [ ]:
emotion_freq = labels_train.sum(axis=0).astype(int)
valid_idx = np.where(valid_mask)[0]

rho_freq, p_freq = spearmanr(emotion_freq[valid_idx], crystal_layer[valid_idx])
rho_depth, p_depth = spearmanr(crystal_layer[valid_idx], max_f1_per_emo[valid_idx])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

ax = axes[0]
ax.scatter(emotion_freq[valid_idx], crystal_layer[valid_idx],
           c=max_f1_per_emo[valid_idx], cmap='RdYlGn', s=100,
           edgecolors='white', linewidth=1.5, zorder=3)
for i in valid_idx:
    ax.annotate(emotion_names[i], (emotion_freq[i], crystal_layer[i]),
                fontsize=7, textcoords='offset points', xytext=(5, 3))
ax.set_xlabel('Training frequency (# examples)', fontsize=12)
ax.set_ylabel('Crystallization layer', fontsize=12)
ax.set_title('Does frequency predict depth?', fontsize=13, fontweight='bold')
ax.text(0.05, 0.95, f'Spearman r = {rho_freq:.3f}\np = {p_freq:.4f}',
        transform=ax.transAxes, fontsize=11, va='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

ax = axes[1]
ax.scatter(crystal_layer[valid_idx], max_f1_per_emo[valid_idx],
           c=emotion_freq[valid_idx], cmap='viridis', s=100,
           edgecolors='white', linewidth=1.5, zorder=3)
for i in valid_idx:
    ax.annotate(emotion_names[i], (crystal_layer[i], max_f1_per_emo[i]),
                fontsize=7, textcoords='offset points', xytext=(5, 3))
ax.set_xlabel('Crystallization layer', fontsize=12)
ax.set_ylabel('Max probe F1', fontsize=12)
ax.set_title('Does depth predict probe quality?', fontsize=13, fontweight='bold')
ax.text(0.05, 0.95, f'Spearman r = {rho_depth:.3f}\np = {p_depth:.4f}',
        transform=ax.transAxes, fontsize=11, va='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'probing_correlations.png'), bbox_inches='tight')
plt.show()

### B.5 — Layer Information Gain

Mean increase in probe F1 relative to the previous layer. Highlights the
layers that do the heavy lifting.

In [ ]:
info_gain = np.zeros(N_HIDDEN_LAYERS)
info_gain[0] = probe_f1[:, 0].mean()
for l in range(1, N_HIDDEN_LAYERS):
    info_gain[l] = (probe_f1[:, l] - probe_f1[:, l - 1]).mean()

fig, ax = plt.subplots(figsize=(12, 6))
bar_colors = ['#3498db' if g >= 0 else '#e74c3c' for g in info_gain]
ax.bar(range(N_HIDDEN_LAYERS), info_gain, color=bar_colors, edgecolor='white', linewidth=1.5)
ax.set_xticks(range(N_HIDDEN_LAYERS))
ax.set_xticklabels(LAYER_LABELS)
ax.set_xlabel('Layer', fontsize=13)
ax.set_ylabel('Mean F1 gain over previous layer', fontsize=13)
ax.set_title('Where BERT Does the Heavy Lifting\n'
             '(mean probe F1 gain per layer, across all emotions)',
             fontsize=14, fontweight='bold')
ax.axhline(y=0, color='black', linewidth=0.5)
for i, g in enumerate(info_gain):
    ax.text(i, g + 0.002 if g >= 0 else g - 0.005, f'{g:+.3f}',
            ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'probing_info_gain.png'), bbox_inches='tight')
plt.show()

print('Top 3 layers by information gain:')
for l in np.argsort(-info_gain)[:3]:
    print(f'  {LAYER_LABELS[l]}: {info_gain[l]:+.4f}')

## Part C — Link to Compression Sensitivity

If emotion X crystallizes at layer L, compressing L should hurt X the most.
Cross-check against the per-layer lesion study from notebook 08.

In [ ]:
lesion_csv = os.path.join(RESULTS_DIR, 'notebook8', 'lesion_per_layer.csv')
if not os.path.exists(lesion_csv):
    lesion_csv = os.path.join(RESULTS_DIR, 'lesion_per_layer.csv')

probe_vs_lesion_df = None
rho_pvl = None
p_pvl = None

if os.path.exists(lesion_csv):
    df_lesion = pd.read_csv(lesion_csv)
    print(f'Loaded lesion results: {len(df_lesion)} rows from {lesion_csv}')

    layer_probe_importance = probe_f1.mean(axis=0)[1:]  # skip embedding
    lesion_retention = []
    for layer_idx in range(12):
        layer_data = df_lesion[df_lesion['layer'] == layer_idx]
        col = 'retention' if 'retention' in layer_data.columns else 'f1'
        lesion_retention.append(layer_data[col].mean())
    lesion_retention = np.array(lesion_retention)

    rho_pvl, p_pvl = spearmanr(layer_probe_importance, lesion_retention)

    probe_vs_lesion_df = pd.DataFrame({
        'layer': range(12),
        'layer_name': [f'L{i}' for i in range(12)],
        'mean_probe_f1': layer_probe_importance,
        'mean_lesion_retention': lesion_retention,
    })

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.scatter(layer_probe_importance, lesion_retention,
               s=150, c=range(12), cmap='viridis',
               edgecolors='white', linewidth=2, zorder=3)
    for l in range(12):
        ax.annotate(f'L{l}', (layer_probe_importance[l], lesion_retention[l]),
                    fontsize=11, fontweight='bold', textcoords='offset points', xytext=(8, 5))
    ax.set_xlabel('Mean probe F1 at layer (probing importance)', fontsize=13)
    ax.set_ylabel('Mean F1 retention under compression (lesion)', fontsize=13)
    ax.set_title(f'Probing vs Compression Sensitivity\n'
                 f'Spearman r = {rho_pvl:.3f}, p = {p_pvl:.4f}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'probing_vs_compression.png'), bbox_inches='tight')
    plt.show()
else:
    print(f'Lesion results not found (tried results/notebook8/ and results/).')
    print('Run notebook 08_emotional_map.ipynb first to enable this comparison.')

## Part D — Export

All results as CSV under `results/notebook4/`:

- `probe_results.csv` — probe F1 matrix (emotions x layers).
- `probe_results_long.csv` — same data in long form.
- `crystallization_layers.csv` — per-emotion summary (crystal layer, max F1, frequency).
- `layer_information_gain.csv` — mean F1 gain per layer.
- `emotion_frequency.csv` — training-set counts.
- `probing_correlations.csv` — Spearman correlations summarised.
- `probe_vs_lesion.csv` — link to the lesion study (if available).
- `probing_summary.csv` — one-row master summary.

In [ ]:
exports = {}

# 1. Probe F1 matrix (wide)
df_probe = pd.DataFrame(probe_f1, index=emotion_names, columns=LAYER_LABELS)
df_probe.index.name = 'emotion'
exports['probe_results.csv'] = df_probe.reset_index()

# 2. Probe F1 matrix (long)
long_rows = []
for emo_idx, emo in enumerate(emotion_names):
    for layer_idx, layer_name in enumerate(LAYER_LABELS):
        long_rows.append({
            'emotion': emo,
            'layer_idx': layer_idx,
            'layer_name': layer_name,
            'probe_f1': probe_f1[emo_idx, layer_idx],
        })
exports['probe_results_long.csv'] = pd.DataFrame(long_rows)

# 3. Crystallization summary
exports['crystallization_layers.csv'] = pd.DataFrame({
    'emotion': emotion_names,
    'crystallization_layer': crystal_layer,
    'crystallization_layer_name': [LAYER_LABELS[int(cl)] for cl in crystal_layer],
    'max_probe_f1': max_f1_per_emo,
    'argmax_layer': probe_f1.argmax(axis=1),
    'train_frequency': emotion_freq,
})

# 4. Layer information gain
exports['layer_information_gain.csv'] = pd.DataFrame({
    'layer_idx': range(N_HIDDEN_LAYERS),
    'layer_name': LAYER_LABELS,
    'mean_probe_f1': probe_f1.mean(axis=0),
    'information_gain': info_gain,
})

# 5. Emotion frequency
exports['emotion_frequency.csv'] = pd.DataFrame({
    'emotion': emotion_names,
    'train_count': emotion_freq,
    'train_fraction': emotion_freq / labels_train.shape[0],
})

# 6. Correlations
exports['probing_correlations.csv'] = pd.DataFrame([
    {'relation': 'frequency_vs_crystal_layer',     'spearman_r': rho_freq,  'p_value': p_freq},
    {'relation': 'crystal_layer_vs_max_probe_f1',  'spearman_r': rho_depth, 'p_value': p_depth},
])

# 7. Probe vs lesion (optional)
if probe_vs_lesion_df is not None:
    exports['probe_vs_lesion.csv'] = probe_vs_lesion_df

# 8. Master summary
summary_rows = [{
    'n_emotions': num_labels,
    'n_layers': N_HIDDEN_LAYERS,
    'n_probes_trained': int(np.count_nonzero(probe_f1)),
    'mean_probe_f1_valid': float(probe_f1[valid_mask].mean()),
    'max_probe_f1_global': float(probe_f1.max()),
    'top_info_gain_layer': LAYER_LABELS[int(np.argmax(info_gain))],
    'top_info_gain_value': float(info_gain.max()),
    'spearman_freq_vs_depth': rho_freq,
    'spearman_depth_vs_f1': rho_depth,
    'spearman_probe_vs_lesion': rho_pvl if rho_pvl is not None else float('nan'),
}]
exports['probing_summary.csv'] = pd.DataFrame(summary_rows)

for fname, df in exports.items():
    out_path = os.path.join(EXPORT_DIR, fname)
    df.to_csv(out_path, index=False)
    print(f'  [{len(df):>5d} rows] {fname}')

print('\n' + '=' * 60)
print('SUMMARY — Phase 4: Probing Classifiers')
print('=' * 60)
print(f'Probes trained: {num_labels} emotions x {N_HIDDEN_LAYERS} layers = {num_labels * N_HIDDEN_LAYERS}')
print(f'Mean probe F1 (valid emotions): {probe_f1[valid_mask].mean():.4f}')
print(f'Top info-gain layer: {LAYER_LABELS[int(np.argmax(info_gain))]} ({info_gain.max():+.4f})')
print(f'\nEarliest crystallizers:')
for emo, cl, mf1 in valid_emos[:5]:
    print(f'  {emo:18s}  crystal: {LAYER_LABELS[int(cl)]:>4s}  max F1: {mf1:.3f}')
print(f'\nLatest crystallizers:')
for emo, cl, mf1 in valid_emos[-5:]:
    print(f'  {emo:18s}  crystal: {LAYER_LABELS[int(cl)]:>4s}  max F1: {mf1:.3f}')
print(f'\nExported {len(exports)} CSVs to {EXPORT_DIR}')